In [1]:
# %% [markdown]
# ============================================================
# example.ipynb — Updated for current repo structure (2026-01)
# ============================================================
# Assumed repo structure (current):
#
# project-root/
# ├── data/
# │   ├── examples/ X.csv, Y.csv
# │   └── raw/ 251205_ParisHotel_comment_score.csv
# ├── notebook/
# │   ├── example.ipynb
# │   └── analysis01.ipynb
# ├── src/
# │   └── BayesianVariableSelection/...
# └── output/   (created by notebook if missing)
#
# Key updates:
# - Robust PROJECT_ROOT detection
# - Quarterly aggregation (locationId × quarter) using TimeBinningConfig(freq="Q")
# - No artificial tokens like [platform=OTHER] injected into review_text
# - Safe word corpus file normalization (2-column format expected by your code)

# %% CELL 1 — Project root + sys.path (run once per kernel)
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection
SRC_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/src


In [2]:
# %% CELL 2 — Imports
import numpy as np
import pandas as pd

# Core model (structured path)
from BayesianVariableSelection.label_bvs import LabelBVS

In [3]:
# %% CELL 3 — Paths
X_PATH = PROJECT_ROOT / "data" / "examples" / "X.csv"
Y_PATH = PROJECT_ROOT / "data" / "examples" / "Y.csv"

print("X_PATH:", X_PATH)
print("Y_PATH:", Y_PATH)

X_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/X.csv
Y_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/Y.csv


In [4]:
# %% CELL 4 — Load X and y
X = pd.read_csv(X_PATH)
y_df = pd.read_csv(Y_PATH)

# If Y.csv has a single column, squeeze into a Series
if y_df.shape[1] == 1:
    y = y_df.iloc[:, 0]
else:
    # If it has multiple columns, you must choose one
    # (Keeping deterministic default: first column)
    y = y_df.iloc[:, 0]

# Basic checks
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y head:")
display(y.head())

# Ensure numeric y
y = pd.to_numeric(y, errors="coerce")
print("y numeric describe:")
print(y.describe())

# Drop rows with missing y (and align X)
mask = ~y.isna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("After dropping NA y:")
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000, 100)
y shape: (1000,)
y head:


0    69.702397
1    59.777245
2    59.765703
3    61.937234
4    54.124244
Name: x, dtype: float64

y numeric describe:
count    1000.000000
mean       45.648723
std        14.945972
min         1.890083
25%        35.137402
50%        45.887801
75%        56.428813
max        95.588017
Name: x, dtype: float64
After dropping NA y:
X shape: (1000, 100)
y shape: (1000,)


In [5]:
# %% CELL 5 — Optional: standardize column names (safe)
# Some pipelines expect clean string column names.
X.columns = [str(c) for c in X.columns]

In [7]:
# %% CELL 6 — Initialize LabelBVS
# NOTE: Parameters vary by implementation; these are common and safe defaults.
# If your LabelBVS signature differs, paste the error and I’ll adjust precisely.

model = LabelBVS(
    X.fillna(0), # X: word frequency matrix
    y, # y: rating 
    dist_type = 'cooccur',
    track = True,
)
print("✔ LabelBVS initialized")

✔ LabelBVS initialized


In [8]:
# %% CELL 7 — Fit (debug-fast settings)
# Total iterations = burnin + iteration
SEGMENTS = 2
ITER     = 500
BURNIN   = 2000
STEP     = 1

# If your LabelBVS uses different parameter names, adjust here.
model.initialize(
    segment_no=SEGMENTS,
    iteration=ITER,
    burnin=BURNIN,
    step=STEP,
)
model.fit()
print("✔ Fit complete")


  0%|          | 0/2500 [00:00<?, ?it/s]

✔ Fit complete


In [9]:
model.visualize_topic_seg(0)

In [10]:
# %% CELL 8 — Basic outputs / diagnostics
# Membership (segment assignment)
membership = model.find_membership()
vals, counts = np.unique(membership, return_counts=True)
print("segment sizes:", dict(zip(vals, counts)))

segment sizes: {np.int64(0): np.int64(599), np.int64(1): np.int64(401)}
